# Indonesian Data Job Market Intelligence
## Pipeline Ekstraksi Data Lowongan Kerja: JobStreet Indonesia (Target Skala 250–500 Loker Bersih)

Notebook ini mengimplementasikan pipeline ekstraksi data lowongan kerja bidang data di portal **JobStreet Indonesia (SEEK Platform)** yang dioptimalkan untuk target **250–500 lowongan unik berkualitas** sesuai panduan proyek (*Panduan Ekstraksi Data Lowongan Kerja*).

### Peningkatan Skala Ekstraksi & Penanganan Duplikasi:
1. **Ekspansi Taksonomi Kata Kunci (12 Peran):** Mengembangkan pencarian dari 5 peran umum menjadi 12 kata kunci komprehensif yang mencakup peran inti (*Core Data*), peran fungsional spesifik (*BI Developer, Analytics Engineer, DBA, ETL, Data Architect*), serta peran berbasis alat dominan (*Power BI, Tableau*). Hal ini meminimalkan tingkat kejenuhan (*search saturation*) dan memperluas penemuan postingan unik.
2. **Paginasi Multi-Halaman Dinamis (*Deep Pagination*):** Menetapkan kuota 75 kartu per kata kunci (menjelajahi halaman 1, 2, hingga 3+ secara otomatis) untuk menjaring lowongan aktif yang tidak hanya berada di halaman pertama.
3. **Penanganan Duplikasi Ketat di Tahap 1 (*Stage-1 Canonical Deduplication*):**
   - Membersihkan URL tracking fragment (`#sol=...`) dan query parameter (`?type=standard`).
   - Ekstraksi `source_job_id` numerik resmi dan deduplikasi lintas kata kunci langsung di Tahap 1 sebelum antrean scraping detail dibentuk, menghemat request dan mencegah data ganda.
4. **Surgical Detail Extraction:** Mengambil deskripsi lengkap dari kontainer resmi `jobAdDetails`, tipe pekerjaan, dan tanggal posting.
5. **Checkpointing Otomatis & Audit Kualitas:** Menyimpan progres ke file CSV setiap 5 lowongan dengan proteksi resume crawling.

### Sel 1: Inisialisasi Dependensi & Pustaka

In [1]:
import os
import sys
import re
import time
import random
import json
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, urlparse

# Konsistensi encoding terminal Windows
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Dependensi JobStreet MVP berhasil dimuat.")

[2026-10-31 22:40:40] Dependensi JobStreet MVP berhasil dimuat.


### Sel 2: Konfigurasi Parameter MVP & Jalur Direktori

In [2]:
# Konfigurasi Path Proyek
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == "extract_data" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# File Checkpoint MVP & Skalasi
CHECKPOINT_LISTINGS_PATH = RAW_DIR / "jobstreet_checkpoint_listings.csv"
CHECKPOINT_DETAILS_PATH = RAW_DIR / "jobstreet_checkpoint_details.csv"

# Kata Kunci Pencarian Komprehensif (Target 250–500 Loker Bersih)
TARGET_KEYWORDS = [
    # Peran Utama (Core Data Roles)
    "Data Analyst",
    "Data Engineer",
    "Data Scientist",
    "Business Intelligence ",
    "Machine Learning Engineer",
    "AI Engineer"
]

# Batas Ekstraksi & Parameter Crawling
MAX_CARDS_PER_KEYWORD = 100  # Target kuota kartu per kata kunci (~2-3 halaman JobStreet)
HEADLESS_MODE = False       # Headless=False untuk observasi visual & mempermudah bypass Cloudflare
MIN_DELAY_SECONDS = 2.2
MAX_DELAY_SECONDS = 4.0

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Raw Data Dir      : {RAW_DIR}")
print(f"Target Keywords   : {len(TARGET_KEYWORDS)} peran ({TARGET_KEYWORDS})")
print(f"Quota per Keyword : {MAX_CARDS_PER_KEYWORD} cards (Potensi pemindaian: {len(TARGET_KEYWORDS) * MAX_CARDS_PER_KEYWORD} kartu)")
print(f"Headless Mode     : {HEADLESS_MODE}")

Project Root      : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia
Raw Data Dir      : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw
Target Keywords   : 6 peran (['Data Analyst', 'Data Engineer', 'Data Scientist', 'Business Intelligence ', 'Machine Learning Engineer', 'AI Engineer'])
Quota per Keyword : 100 cards (Potensi pemindaian: 600 kartu)
Headless Mode     : False


### Sel 3: Fungsi Helper Utilitas & Deduplikasi Kanonikal

In [3]:
def text_or_none(element, separator=" ", strip=True):
    # Mengekstrak teks dari elemen BeautifulSoup dengan penanganan None aman
    if element is None:
        return None
    text = element.get_text(separator=separator, strip=strip)
    return text if text else None


def clean_jobstreet_url_and_id(raw_url: str) -> tuple[str, str]:
    # Membersihkan URL JobStreet dari parameter tracking (?type=standard, #sol=...)
    # dan mengekstrak source_job_id numerik kanonikal.
    if not raw_url:
        return "", ""
    if not raw_url.startswith("http"):
        raw_url = urljoin("https://id.jobstreet.com", raw_url)
    parsed = urlparse(raw_url)
    clean = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
    
    match_id = re.search(r"/job/(\d+)", clean)
    source_job_id = match_id.group(1) if match_id else ""
    return clean, source_job_id


def save_checkpoint(records: list, file_path: Path):
    # Menyimpan progres ekstraksi ke file CSV sebagai checkpoint data mentah.
    if not records:
        return
    df = pd.DataFrame(records)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Checkpoint diperbarui: {len(df)} baris tersimpan di {file_path.name}")

### Sel 4: Inisialisasi WebDriver Selenium (Anti-Detection Mode)

In [4]:
def create_driver(headless: bool = False) -> webdriver.Chrome:
    # Menginisialisasi WebDriver Chrome dengan konfigurasi anti-deteksi bot untuk JobStreet
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--window-size=1366,768")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(35)
    return driver

driver = create_driver(headless=HEADLESS_MODE)
print(f"[{datetime.now().strftime('%H:%M:%S')}] Chrome WebDriver aktif (Headless={HEADLESS_MODE}).")

[22:40:41] Chrome WebDriver aktif (Headless=False).


### Sel 5: Tahap 1 – Parser Daftar Kartu Lowongan (*Listing Cards*) dengan Deduplikasi

In [5]:
def parse_listing_card(article_tag, keyword: str) -> dict:
    # Mengekstrak metadata dari elemen kartu lowongan (<article>) pada JobStreet
    title_el = article_tag.find("a", attrs={"data-automation": "jobTitle"})
    company_el = article_tag.find("a", attrs={"data-automation": "jobCompany"})
    location_el = article_tag.find("a", attrs={"data-automation": "jobLocation"})
    salary_el = article_tag.find("span", attrs={"data-automation": "jobSalary"})
    
    raw_href = title_el.get("href", "") if title_el else ""
    job_url, source_job_id = clean_jobstreet_url_and_id(raw_href)
    
    return {
        "source_job_id": source_job_id,
        "job_url": job_url,
        "keyword": keyword,
        "title_raw": text_or_none(title_el),
        "company_raw": text_or_none(company_el),
        "location_raw": text_or_none(location_el),
        "salary_raw": text_or_none(salary_el),
        "extracted_at": datetime.now().isoformat()
    }


def extract_jobstreet_listings(driver, keyword: str, max_cards: int = 75) -> list:
    # Mengumpulkan kartu lowongan dari JobStreet untuk satu kata kunci dengan paginasi
    slug = keyword.lower().strip().replace(" ", "-")
    listings = []
    seen_in_kw = set()
    page = 1
    
    print(f"\n--- Mengambil listing untuk keyword: '{keyword}' (Target: {max_cards}) ---")
    
    while len(listings) < max_cards:
        search_url = f"https://id.jobstreet.com/id/{slug}-jobs?page={page}"
        print(f"Membuka halaman {page}: {search_url}")
        
        try:
            driver.get(search_url)
            WebDriverWait(driver, 15).until(lambda d: len(d.find_elements(By.TAG_NAME, "article")) > 0)
            time.sleep(random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))
        except Exception as e:
            print(f"Peringatan: Gagal memuat halaman {page} untuk '{keyword}': {e}")
            break
            
        soup = BeautifulSoup(driver.page_source, "html.parser")
        articles = soup.find_all("article")
        if not articles:
            print(f"Tidak ada kartu lowongan lagi pada halaman {page}.")
            break
            
        initial_count = len(listings)
        for art in articles:
            card_data = parse_listing_card(art, keyword)
            jid = card_data.get("source_job_id")
            url = card_data.get("job_url")
            if jid and url and ("/job/" in url) and (jid not in seen_in_kw):
                seen_in_kw.add(jid)
                listings.append(card_data)
                if len(listings) >= max_cards:
                    break
                    
        new_on_page = len(listings) - initial_count
        print(f"Halaman {page}: Mendapatkan {new_on_page} kartu baru (Total: {len(listings)}/{max_cards})")
        
        if new_on_page == 0:
            print("Tidak ada kartu baru ditemukan pada halaman ini. Selesai.")
            break
            
        page += 1
        time.sleep(random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))
        
    return listings

### Sel 6: Tahap 2 – Parser Detail Lowongan (*Surgical Detail Parser*)

In [6]:
def parse_jobstreet_detail(html: str, job_url: str, source_job_id: str = None) -> dict:
    # Mengekstrak detail lowongan kerja secara presisi berdasarkan DOM JobStreet
    soup = BeautifulSoup(html, "html.parser")
    
    title_el = soup.find("h1", attrs={"data-automation": "job-detail-title"})
    company_el = soup.find("span", attrs={"data-automation": "advertiser-name"})
    location_el = soup.find("span", attrs={"data-automation": "job-detail-location"})
    salary_el = soup.find("span", attrs={"data-automation": "job-detail-salary"})
    work_type_el = soup.find("span", attrs={"data-automation": "job-detail-work-type"})
    
    classification_els = soup.find_all("a", attrs={"data-automation": "job-detail-classifications"})
    classifications = [c.get_text(strip=True) for c in classification_els if c.get_text(strip=True)]
    classification_raw = " | ".join(classifications) if classifications else None
    
    desc_el = soup.find("div", attrs={"data-automation": "jobAdDetails"})
    description_raw = desc_el.get_text(separator="\n", strip=True) if desc_el else None
    
    posted_date_raw = None
    for span in soup.find_all("span"):
        text = span.get_text(strip=True)
        if re.search(r"\b(?:diposting|posted)\b", text, re.IGNORECASE):
            posted_date_raw = text
            break
            
    parse_status = "OK" if description_raw else "EMPTY_DESCRIPTION"
    parse_error = None if description_raw else "Kontainer jobAdDetails tidak ditemukan atau kosong."
    
    return {
        "source_job_id": source_job_id,
        "job_url": job_url,
        "portal": "JobStreet",
        "title_raw": text_or_none(title_el),
        "company_raw": text_or_none(company_el),
        "location_raw": text_or_none(location_el),
        "salary_raw": text_or_none(salary_el),
        "job_type_raw": text_or_none(work_type_el),
        "industry_raw": classification_raw,
        "posted_date_raw": posted_date_raw,
        "description_raw": description_raw,
        "detail_extracted_at": datetime.now().isoformat(),
        "parse_status": parse_status,
        "parse_error": parse_error
    }

### Sel 7: Eksekusi Pipeline Ekstraksi MVP dengan Deduplikasi Ketat

In [7]:
# ==========================================
# 1. Kumpulkan daftar lowongan unik (Tahap 1)
# ==========================================
all_listings = []
seen_job_ids = set()
seen_urls = set()
duplicates_skipped = 0

# Muat checkpoint listing jika sudah ada
if CHECKPOINT_LISTINGS_PATH.exists():
    df_existing_listings = pd.read_csv(CHECKPOINT_LISTINGS_PATH)
    if 'source_job_id' in df_existing_listings:
        seen_job_ids.update(df_existing_listings['source_job_id'].dropna().astype(str).tolist())
    seen_urls.update(df_existing_listings['job_url'].dropna().tolist())
    all_listings.extend(df_existing_listings.to_dict('records'))
    print(f"Melanjutkan dari checkpoint listing: {len(seen_job_ids)} lowongan tersimpan.")

for kw in TARGET_KEYWORDS:
    cards = extract_jobstreet_listings(driver, kw, max_cards=MAX_CARDS_PER_KEYWORD)
    new_cards = []
    for c in cards:
        jid = str(c.get('source_job_id', ''))
        jurl = c.get('job_url', '')
        if jid in seen_job_ids or jurl in seen_urls:
            duplicates_skipped += 1
            continue
        seen_job_ids.add(jid)
        seen_urls.add(jurl)
        all_listings.append(c)
        new_cards.append(c)
        
    print(f"Kata kunci '{kw}': +{len(new_cards)} lowongan baru ditambahkan.")
    if new_cards:
        save_checkpoint(all_listings, CHECKPOINT_LISTINGS_PATH)
    time.sleep(random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))

df_listings = pd.DataFrame(all_listings)
print(f"\nTotal listing unik terdeduplikasi: {len(df_listings)} (Duplikasi lintas keyword dicegah: {duplicates_skipped}).")

# ==========================================
# 2. Ambil detail setiap lowongan (Tahap 2)
# ==========================================
details_records = []
processed_ids = set()

if CHECKPOINT_DETAILS_PATH.exists():
    df_existing_details = pd.read_csv(CHECKPOINT_DETAILS_PATH)
    if 'source_job_id' in df_existing_details:
        processed_ids.update(df_existing_details['source_job_id'].dropna().astype(str).tolist())
    details_records.extend(df_existing_details.to_dict('records'))
    print(f"Melanjutkan dari checkpoint detail: {len(processed_ids)} lowongan sudah diproses.")

# Ambil lowongan yang belum diproses
pending_listings = df_listings[~df_listings['source_job_id'].astype(str).isin(processed_ids)]
total_queue = len(pending_listings)
print(f"Antrean detail yang akan diambil: {total_queue} URL.\n")

for idx, row in pending_listings.reset_index(drop=True).iterrows():
    url = row['job_url']
    job_id = row.get('source_job_id')
    print(f"[{idx+1}/{total_queue}] Mengambil: {url}")
    
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, "div[data-automation='jobAdDetails'], h1[data-automation='job-detail-title']")) > 0
        )
        time.sleep(random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))
        
        detail_data = parse_jobstreet_detail(driver.page_source, url, source_job_id=job_id)
        for field in ["title_raw", "company_raw", "location_raw", "salary_raw"]:
            if not detail_data.get(field) and row.get(field):
                detail_data[field] = row[field]
        detail_data["keyword"] = row.get("keyword")
        
        details_records.append(detail_data)
        processed_ids.add(str(job_id))
    except Exception as e:
        print(f"  [ERROR] Gagal memproses {url}: {e}")
        details_records.append({
            "source_job_id": job_id,
            "job_url": url,
            "keyword": row.get("keyword"),
            "portal": "JobStreet",
            "title_raw": row.get("title_raw"),
            "company_raw": row.get("company_raw"),
            "location_raw": row.get("location_raw"),
            "salary_raw": row.get("salary_raw"),
            "job_type_raw": None,
            "industry_raw": None,
            "posted_date_raw": None,
            "description_raw": None,
            "detail_extracted_at": datetime.now().isoformat(),
            "parse_status": "EXCEPTION",
            "parse_error": str(e)
        })
        
    if (idx + 1) % 5 == 0 or (idx + 1) == total_queue:
        save_checkpoint(details_records, CHECKPOINT_DETAILS_PATH)
    time.sleep(random.uniform(1.0, 2.0))

df_raw = pd.DataFrame(details_records)
print(f"\nTahap ekstraksi JobStreet MVP selesai. Total data mentah: {len(df_raw)} lowongan.")


--- Mengambil listing untuk keyword: 'Data Analyst' (Target: 100) ---
Membuka halaman 1: https://id.jobstreet.com/id/data-analyst-jobs?page=1
Halaman 1: Mendapatkan 30 kartu baru (Total: 30/100)
Membuka halaman 2: https://id.jobstreet.com/id/data-analyst-jobs?page=2
Halaman 2: Mendapatkan 30 kartu baru (Total: 60/100)
Membuka halaman 3: https://id.jobstreet.com/id/data-analyst-jobs?page=3
Halaman 3: Mendapatkan 30 kartu baru (Total: 90/100)
Membuka halaman 4: https://id.jobstreet.com/id/data-analyst-jobs?page=4
Peringatan: Gagal memuat halaman 4 untuk 'Data Analyst': Message: 

Kata kunci 'Data Analyst': +90 lowongan baru ditambahkan.
[22:41:24] Checkpoint diperbarui: 90 baris tersimpan di jobstreet_checkpoint_listings.csv

--- Mengambil listing untuk keyword: 'Data Engineer' (Target: 100) ---
Membuka halaman 1: https://id.jobstreet.com/id/data-engineer-jobs?page=1
Halaman 1: Mendapatkan 30 kartu baru (Total: 30/100)
Membuka halaman 2: https://id.jobstreet.com/id/data-engineer-jobs?pa

### Sel 8: Audit Kualitas Ekstraksi Data Mentah (`audit_extraction`)

In [8]:
def audit_extraction(df: pd.DataFrame) -> pd.Series:
    # Menghasilkan ringkasan audit kualitas data hasil ekstraksi
    total = len(df)
    summary = {
        "total_rows": total,
        "missing_title": df["title_raw"].isna().sum() if "title_raw" in df else 0,
        "missing_company": df["company_raw"].isna().sum() if "company_raw" in df else 0,
        "missing_location": df["location_raw"].isna().sum() if "location_raw" in df else 0,
        "missing_salary": df["salary_raw"].isna().sum() if "salary_raw" in df else 0,
        "missing_description": df["description_raw"].isna().sum() if "description_raw" in df else 0,
        "missing_posted_date": df["posted_date_raw"].isna().sum() if "posted_date_raw" in df else 0,
        "parse_success_count": (df["parse_status"] == "OK").sum() if "parse_status" in df else 0,
        "duplicate_urls": df["job_url"].duplicated().sum() if "job_url" in df else 0,
        "duplicate_job_ids": df["source_job_id"].dropna().duplicated().sum() if "source_job_id" in df else 0
    }
    return pd.Series(summary)

print("=== RINGKASAN AUDIT KUALITAS EKSTRAKSI JOBSTREET MVP ===")
audit_result = audit_extraction(df_raw)
display(audit_result.to_frame(name="Nilai"))

print("\nDistribusi Kata Kunci:")
display(df_raw["keyword"].value_counts().to_frame(name="Jumlah"))

print("\nSampel 5 Baris Data Mentah:")
display(df_raw[["source_job_id", "title_raw", "company_raw", "location_raw", "salary_raw", "job_type_raw", "parse_status"]].head(5))

=== RINGKASAN AUDIT KUALITAS EKSTRAKSI JOBSTREET MVP ===


,Nilai
total_rows,189
missing_title,0
missing_company,0
missing_location,0
missing_salary,135
missing_description,0
missing_posted_date,0
parse_success_count,189
duplicate_urls,0
duplicate_job_ids,0



Distribusi Kata Kunci:


,Jumlah
keyword,
Data Analyst,90
Data Engineer,43
Machine Learning Engineer,38
Business Intelligence,18



Sampel 5 Baris Data Mentah:


,source_job_id,title_raw,company_raw,location_raw,salary_raw,job_type_raw,parse_status
0,95292954,Data Analyst,PT Prima Nusantara Permai,"Surabaya, Jawa Timur",Rp 5.288.800 – Rp 7.500.000 per month,Full time,OK
1,95570805,Data Analyst Operational Staff (E-commerce),JINISO.ID,"Pagedangan, Banten",None,Full time,OK
2,95359365,Data Insight Specialist,"PT. BFI FINANCE INDONESIA, Tbk","Tangerang District, Banten",None,Full time,OK
3,95225815,DATA ANALYST,Pengiklan Anonim,"Jakarta Timur, Jakarta Raya",None,Full time,OK
4,95554234,JUNIOR DATA ANALYST,PT Smart Multi Finance,"Kelapa Gading, Jakarta Raya",None,Full time,OK


### Sel 9: Ekspor Dataset Mentah Final & Penutupan WebDriver

In [9]:
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_csv_path = RAW_DIR / f"jobstreet_raw_{timestamp_str}.csv"
raw_json_path = RAW_DIR / f"jobstreet_raw_{timestamp_str}.json"

df_raw.to_csv(raw_csv_path, index=False, encoding="utf-8")
df_raw.to_json(raw_json_path, orient="records", indent=2, force_ascii=False)

print(f"Data mentah final JobStreet berhasil diekspor:")
print(f"  - CSV  : {raw_csv_path.resolve()} ({raw_csv_path.stat().st_size / 1024:.1f} KB)")
print(f"  - JSON : {raw_json_path.resolve()} ({raw_json_path.stat().st_size / 1024:.1f} KB)")

try:
    driver.quit()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] WebDriver berhasil ditutup.")
except Exception as e:
    print(f"Catatan penutupan driver: {e}")

Data mentah final JobStreet berhasil diekspor:
  - CSV  : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw\jobstreet_raw_20261031_230119.csv (460.6 KB)
  - JSON : C:\Users\Victus 16\data-project\analisi_loker_data_indonesia\data\raw\jobstreet_raw_20261031_230119.json (527.3 KB)
[23:01:22] WebDriver berhasil ditutup.
